## 1. Defining cleaning criteria
The exploration notebook identified several data-quality issues that need to be addressed before the analysis.  
The cleaning criteria are:  
- Target vessel types: retain Cargo, Tanker, and Passenger vessels only  
- Valid vessel identity: investigate and remove malformed or non-vessel MMSI records  
- Present and realistic SOG values: remove missing speeds and investigate physically impossible speeds  
- Relevant navigational status: exclude observations where vessels are stationary or otherwise unsuitable for measuring voyage progress, such as vessels at anchor or moored  
- Valid destination: exclude unsuitable destinations such as 'Unknown' and 'FOR ORDERS'  
- Destination normalization: standardize different names and codes that refer to the same port  
- ETA consistency: ensure reported ETA values are parseable and logically valid relative to each observation timestamp and voyage information  

## Target Vessel Types
We simply keep Cargo, Tanker and passenger vessels

In [ ]:
import pandas as pd

cols = ['MMSI', 'Latitude', 'Longitude', 'SOG', 'Ship type', 'Destination', 'ETA', '# Timestamp', 'Type of mobile', 'Navigational status', 'Name']

df_ais = pd.read_csv('../data/raw/aisdk-2024-08-07.csv', usecols=cols)

df_target = df_ais[df_ais['Ship type'].isin(['Cargo', 'Tanker', 'Passenger'])]
rows_before = df_ais.shape[0]
rows_target = df_target.shape[0]

print(rows_before, rows_target)

We can see that just by filtering out vessel types we are not interested in we get back 11,842,463 observations/rows of data from the original 28,203,137

## Present And Realistic SOG Values
Missing and physically implausible speeds are further investigated and removed

In [2]:
pd.set_option('display.float_format', lambda x: f"{x:.2f}")
print(df_target['SOG'].describe())

count   11823317.00
mean           9.95
std            5.60
min            0.00
25%            7.70
50%           10.90
75%           13.60
max           99.70
Name: SOG, dtype: float64


As we saw in the exploration notebook there is a max value of 99.7 knots. Lets dig deeper to investigate the vessel involved


In [ ]:
speed_anomalies = df_target[df_target['SOG'] > 40]
print(speed_anomalies.shape[0])
print(speed_anomalies['MMSI'].nunique())
print(speed_anomalies.groupby('MMSI')['SOG'].max())
print(speed_anomalies[speed_anomalies['MMSI'] == 229191000])
print(speed_anomalies[speed_anomalies['MMSI'] == 352179000 ])

These two vessels involved contain inaccurate all day records with one being labelled as 'At Anchor' while having SOG values over 50 knots and the other one having physically implausible SOG values. Based on these findings we'll remove them and continue investigating for any other suspicious speeds that could potentially affect data quality.

In [ ]:
df_target = df_target[(df_target['MMSI'] != 229191000) & (df_target['MMSI'] != 352179000)]
speed_anomalies = df_target[(df_target['SOG'] > 30) & (df_target['SOG'] < 40)]
print(speed_anomalies.shape[0])
print(speed_anomalies['MMSI'].nunique())
print(speed_anomalies.groupby('MMSI')['SOG'].max())
print(speed_anomalies[speed_anomalies['MMSI'] == 230712000])
print(speed_anomalies[speed_anomalies['MMSI'] == 257182700])

Upon further inspection of vessels with SOG values within the 30-40kn range we've found two different cases. A cargo vessel with reported SOG values around 30-32kn while its coordinates remained unchanged, indicating unreliable movement. The second one is a passenger vessel with reported SOG values between 30 and 36kn and normal position movement. Despite its movement appearing plausible it lacks ETA data deeming it unfit for our analysis. In conclusion both vessels found in the 30-40kn range will be excluded.

In [5]:
df_target = df_target[(df_target['MMSI'] != 230712000) & (df_target['MMSI'] != 257182700)]

## Relevant Navigational Status
Observations where vessels are stationary/unsuitable for measuring voyage progress will be removed. 
!important distinction: we only exlcude individual observations and not whole MMSIs because a vessel can be stationary for a part of its voyage and moving for another therefore deeming it suitable for voyage progress tracking

In [ ]:
print(df_target['Navigational status'].value_counts())
stationary_stat = ['Under way using engine', 'Constrained by her draught', 'Under way sailing', 'Restricted maneuverability']
df_target = df_target[df_target['Navigational status'].isin(stationary_stat)]

## Valid Vessel Identity
Observations with malformed or non-vessel MMSIs will be investigated and removed. Standard class A vessel MMSIs have 9 digits

In [ ]:
df_target['MMSI'] = df_target['MMSI'].astype(str)
print(df_target['MMSI'].str.len().value_counts())

Our previous finding from the data exploration notebook that some MMSIs were malformed and non-standard turned out to be false. The vessel type filtering we did above removed all the weird base station non-vessel MMSIs. Another useful check would be to see if each MMSI corresponds to a single vessel

In [ ]:
df_target.groupby('MMSI')['Name'].nunique().sort_values(ascending=False)
print(df_target['Name'][df_target['MMSI'] == '218795000'])
print(df_target.loc[df_target['MMSI'] == '218795000', 'Name'].nunique())



MMSI
218795000    2
212499000    2
636022640    1
636022452    1
636022348    1
            ..
209764000    1
209864000    1
209867000    1
636022780    1
211305130    0
Name: Name, Length: 1056, dtype: int64
246102      ROBIN HOOD
248042      ROBIN HOOD
252850      ROBIN HOOD
257755      ROBIN HOOD
262626      ROBIN HOOD
               ...    
26789392    ROBIN HOOD
26814548    ROBIN HOOD
26819320    ROBIN HOOD
26936903    ROBIN HOOD
27030770    ROBIN HOOD
Name: Name, Length: 23297, dtype: object
2
